In [2]:
# 6c_nan_report.ipynb
#
# Reports NaN and negative-value counts per column in the synthetic population parquet.
# Also cross-checks which config-defined feature columns are present / missing.

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config_paths as _cp
importlib.reload(_cp)
from data_pipeline.config_paths import DATA_FOLDER
import data_pipeline.config_variables as _cv
_cv.reload_config_variables()
from data_pipeline.config_variables import VARIABLES
from data_pipeline.config_cluster import WAVE

import pandas as pd
import numpy  as np
from pathlib import Path

# ── Load ──────────────────────────────────────────────────────────────────────
SYNPOP_PARQUET = Path(f"../{DATA_FOLDER}/6_synthetic_population/synthetic_population.parquet")
if not SYNPOP_PARQUET.exists():
    raise FileNotFoundError(f"{SYNPOP_PARQUET} — run 6a_synthetic_population.ipynb first.")

print(f"Loading {SYNPOP_PARQUET} ...")
df = pd.read_parquet(SYNPOP_PARQUET)
n_rows, n_cols = len(df), len(df.columns)
print(f"  {n_rows:,} rows x {n_cols} columns")

# ── Config vs parquet cross-check ─────────────────────────────────────────────
expected = {f"{WAVE}_{b}" for b in VARIABLES}
missing  = sorted(expected - set(df.columns))
extra    = sorted(set(df.columns) - expected)

print(f"\nConfig feature columns ({WAVE}_*): {len(expected)}")
print(f"  Present : {len(expected) - len(missing)}")
if missing:
    print(f"  Missing : {len(missing)}")
    for c in missing:
        print(f"    x {c}")
print(f"  Extra (geography / OHE etc.): {len(extra)}")
for c in extra:
    print(f"    + {c}")

# ── NaN report ────────────────────────────────────────────────────────────────
print("\n-- NaN report --")
nan_counts = df.isna().sum()
nan_cols   = nan_counts[nan_counts > 0].sort_values(ascending=False)
if nan_cols.empty:
    print("No NaNs in any column.")
else:
    print(f"{len(nan_cols)} column(s) with NaNs:")
    display(pd.DataFrame({
        "nan_count":   nan_cols,
        "nan_percent": (nan_cols / n_rows * 100).round(2),
    }).rename_axis("column"))

# ── Negative-value report ─────────────────────────────────────────────────────
print("\n-- Negative-value report --")
num_cols   = df.select_dtypes(include=[np.number]).columns
neg_counts = df[num_cols].apply(lambda s: (s < 0).sum())
neg_cols   = neg_counts[neg_counts > 0].sort_values(ascending=False)
if neg_cols.empty:
    print(f"No negative values in any of the {len(num_cols)} numeric columns.")
else:
    print(f"{len(neg_cols)} column(s) with negative values:")
    display(pd.DataFrame({
        "neg_count":   neg_cols,
        "neg_percent": (neg_cols / n_rows * 100).round(2),
        "min_value":   df[neg_cols.index].min().round(4),
    }).rename_axis("column"))



🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  
  FOUR-LA SUBSET ACTIVE — Newham, Tower Hamlets, Islington, Hounslow only (4 LAs)
  Set USE_FOUR_LA_SUBSET = False for all London or full UK.
🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  

Loading ../data/6_synthetic_population/synthetic_population.parquet ...
  525,150 rows x 70 columns

Config feature columns (o_*): 73
  Present : 41
  Missing : 32
    x o_benbase4
    x o_browse
    x o_drive
    x o_email
    x o_ethn_dv
    x o_fimnsben_dv
    x o_fuelcost
    x o_hcondncode38
    x o_hcondncode96
    x o_hhtype_dv
    x o_locsera
    x o_marstat_dv
    x o_ncars
    x o_othben1
    x o_othben2
    x o_othben6
    x o_othben8
    x o_pcarown
    x o_pidp
    x o_smlook
    x o_smpost
    x o_smtphone
    x o_streaming
    x o_tenure_dv
    x o_traccess
    x o_transp_diff
    x o_trbikefq
    x o_trbusfq
    x o_trcarfq
    x o_trcost
    x o_trtrnfq
    x o_walkfre

,nan_count,nan_percent
column,,
o_payo_dv,326576,62.19
o_derived_work_status,183590,34.96
o_locsere,68714,13.08
o_locserc,61342,11.68
o_locserd,61041,11.62
o_hiquao_dv,57909,11.03
o_nbrsnci_dv,43512,8.29
o_onlinebank,11958,2.28
o_onlinebuy,11958,2.28



-- Negative-value report --
23 column(s) with negative values:


,neg_count,neg_percent,min_value
column,,,
o_mreason1,524351,99.85,-8.0
o_macob,524190,99.82,-8.0
o_pacob,524190,99.82,-8.0
o_wktrvfar,474858,90.42,-8.0
o_plbornc,440103,83.81,-8.0
o_citzn3,430255,81.93,-8.0
o_citzn1,430255,81.93,-8.0
o_citzn2,430255,81.93,-8.0
o_jbttwt,317949,60.54,-8.0
